# CNN & Computer Vision: Exercises

This notebook accompanies the `05_cnn_computer_vision` module. You will work
with the from-scratch NumPy CNN and a synthetic **warung shelf** dataset: tiny
16x16 greyscale images of product packs photographed by an SPG camera.

## Learning objectives

- Inspect the synthetic product images and their class structure.
- Apply convolution kernels by hand and see what `im2col` does.
- Train `ShelfCNN` and read its loss curve and confusion matrix.
- Inspect the learned convolution filters and feature maps.
- Measure how a translation shift affects a CNN compared with dense models.
- Experiment with learning rate, architecture and an IoU helper for detection.

All code cells run on the CPU. The default training run is short enough to keep
the whole notebook well under two minutes.

In [ ]:
import os
import sys
import time

import matplotlib.pyplot as plt
import numpy as np

# Make the module importable whether the notebook is launched from this
# directory or from the repository root.
candidates = [
    os.path.abspath("."),
    os.path.join(os.path.abspath("."), "04_advanced_topics", "05_cnn_computer_vision"),
]
MODULE_DIR = next(
    path for path in candidates
    if os.path.exists(os.path.join(path, "implementation.py"))
)
sys.path.insert(0, MODULE_DIR)

from implementation import (
    PRODUCT_CLASS_NAMES,
    Conv2D,
    Dense,
    Flatten,
    MaxPool2D,
    ReLU,
    Sequential,
    ShelfCNN,
    conv2d,
    generate_shelf_images,
    im2col,
    shift_images,
)

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.dpi"] = 100
RANDOM_STATE = 42

print("Module directory:", MODULE_DIR)
print("Classes:", PRODUCT_CLASS_NAMES)

## 1. Explore the synthetic warung-shelf data

`generate_shelf_images` draws one of four simple shapes on a noisy shelf
background:

| Class | Product | Shape |
|-------|---------|-------|
| 0 | box pack | filled rectangle with a label band |
| 1 | bottle | tall thin body plus a cap |
| 2 | sachet strip | three horizontal bars |
| 3 | can | filled circle with a rim |

Position, size and brightness are jittered. Labels are balanced by construction.

In [ ]:
X_train, y_train = generate_shelf_images(
    n_samples=400, image_size=16, n_classes=4, noise=0.05,
    random_state=RANDOM_STATE,
)
X_test, y_test = generate_shelf_images(
    n_samples=160, image_size=16, n_classes=4, noise=0.05,
    random_state=RANDOM_STATE + 1,
)

print("X_train:", X_train.shape, X_train.dtype)
print("X_test :", X_test.shape, X_test.dtype)
print("Pixel range:", float(X_train.min()), "to", float(X_train.max()))
print("Train class counts:", np.bincount(y_train))
print("Test class counts :", np.bincount(y_test))

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for row in range(2):
    for col in range(4):
        idx = row * 4 + col
        ax = axes[row, col]
        ax.imshow(X_test[idx, 0], cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"label={y_test[idx]} ({PRODUCT_CLASS_NAMES[y_test[idx]]})",
                     fontsize=9)
        ax.axis("off")
plt.suptitle("Synthetic SPG camera crops (16x16)")
plt.tight_layout()
plt.show()

## 2. Convolution by hand

A convolution kernel slides over the image and produces a feature map. Edge
kernels respond strongly where intensity changes; a box blur smooths the image.
Try changing the kernels below and re-running the cell.

In [ ]:
image = X_test[0, 0]

vertical_edge = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1],
], dtype=float)

horizontal_edge = np.array([
    [-1, -1, -1],
    [0, 0, 0],
    [1, 1, 1],
], dtype=float)

blur = np.ones((3, 3), dtype=float) / 9.0

vertical_map = conv2d(image, vertical_edge, stride=1, padding=1)
horizontal_map = conv2d(image, horizontal_edge, stride=1, padding=1)
blurred = conv2d(image, blur, stride=1, padding=1)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, data, title in zip(
    axes,
    [image, vertical_map, horizontal_map, blurred],
    ["Original", "Vertical edges", "Horizontal edges", "Box blur"],
):
    ax.imshow(data, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

print("Input shape :", image.shape)
print("Output shape:", vertical_map.shape)

## 3. `im2col`: turning convolution into a matrix product

`im2col` extracts every receptive field into a row. Convolution then becomes one
matrix multiplication, which is much faster than Python loops.

**Exercise:** change `kernel_size`, `stride` and `padding` below. Before running,
predict the number of rows and columns using

```
out_h = (H + 2P - k_h) / S + 1
out_w = (W + 2P - k_w) / S + 1
rows  = N * out_h * out_w
cols  = C * k_h * k_w
```

In [ ]:
kernel_size = 3
stride = 2
padding = 1

cols = im2col(X_train[:8], kernel_size=kernel_size, stride=stride, padding=padding)
out_h = (16 + 2 * padding - kernel_size) // stride + 1
out_w = out_h  # square images and kernel

print("cols shape:", cols.shape)
print("expected  :", (8 * out_h * out_w, 1 * kernel_size * kernel_size))

# The adjoint property: <im2col(x), r> == <x, col2im(r)>.
from implementation import col2im

rng = np.random.default_rng(0)
r = rng.normal(size=cols.shape)
back = col2im(r, X_train[:8].shape, kernel_size, stride, padding)
print("adjoint difference:", abs(np.sum(cols * r) - np.sum(X_train[:8] * back)))

## 4. Train the CNN

The default `ShelfCNN` has two convolution blocks:

```
Conv(1 -> 8, 3x3) -> ReLU -> MaxPool(2) -> 8x8
Conv(8 -> 16, 3x3) -> ReLU -> MaxPool(2) -> 4x4
Flatten -> Dense(256 -> 64) -> ReLU -> Dense(64 -> 4)
```

Training uses mini-batch SGD with momentum. Watch the loss fall and compare
train/test accuracy.

In [ ]:
model = ShelfCNN(n_classes=4, image_size=16, random_state=RANDOM_STATE)

started = time.perf_counter()
history = model.fit(
    X_train,
    y_train,
    epochs=12,
    batch_size=32,
    lr=0.02,
    momentum=0.9,
    verbose=0,
)
elapsed = time.perf_counter() - started

print(f"Training time: {elapsed:.2f}s")
print(f"Loss: {history[0]:.4f} -> {history[-1]:.4f}")
print(f"Train accuracy: {model.score(X_train, y_train):.4f}")
print(f"Test accuracy : {model.score(X_test, y_test):.4f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(history) + 1), history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Mean cross-entropy loss")
plt.title("ShelfCNN training loss")
plt.grid(True)
plt.show()

## 5. Confusion matrix and per-class accuracy

A single accuracy number can hide class-specific failures. The rows of the
confusion matrix are true classes and the columns are predictions.

In [ ]:
predictions = model.predict(X_test)

cm = np.zeros((4, 4), dtype=int)
for true_label, pred_label in zip(y_test, predictions):
    cm[true_label, pred_label] += 1

print("Confusion matrix (rows=true, cols=pred):")
print(cm)

print("\nPer-class accuracy:")
for c, name in enumerate(PRODUCT_CLASS_NAMES):
    mask = y_test == c
    print(f"  {c} {name:<13}: {np.mean(predictions[mask] == y_test[mask]):.3f} "
          f"({mask.sum()} samples)")

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set_xticklabels(PRODUCT_CLASS_NAMES, rotation=30, ha="right")
ax.set_yticklabels(PRODUCT_CLASS_NAMES)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Test-set confusion matrix")
for i in range(4):
    for j in range(4):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

## 6. Inspect the learned filters and feature maps

The first `Conv2D` layer has weights of shape `(out_ch, in_ch, k_h, k_w)`.
Even on tiny synthetic data, different filters specialise for edges, bars and
circular boundaries.

In [ ]:
first_conv = next(layer for layer in model.layers if isinstance(layer, Conv2D))
filters = first_conv.params["W"]
print("First conv weight shape:", filters.shape)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    kernel = filters[i, 0]
    ax.imshow(kernel, cmap="RdBu_r")
    ax.set_title(f"filter {i}")
    ax.axis("off")
plt.suptitle("Learned 3x3 filters of the first convolution")
plt.tight_layout()
plt.show()

In [ ]:
# Feature maps produced by the first convolution for one test image.
sample = X_test[:1]
feature_maps = first_conv.forward(sample)
print("Feature map shape:", feature_maps.shape)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(feature_maps[0, i], cmap="viridis")
    ax.set_title(f"channel {i}")
    ax.axis("off")
plt.suptitle("First-layer feature maps")
plt.tight_layout()
plt.show()

# Restore a clean forward cache for the next prediction call.
model.predict_proba(X_test[:1])

## 7. Translation shift: the CNN inductive bias

Move every test image two pixels down and two pixels right. Dense models on
flattened pixels see a completely different input vector. A CNN shares filters
across positions and pools local evidence, so it should degrade more gracefully.
Run `sklearn_example.py` for the full comparison with an MLP and logistic
regression.

In [ ]:
X_shifted = shift_images(X_test, shift_y=2, shift_x=2, fill=0.15)

clean_acc = model.score(X_test, y_test)
shifted_acc = model.score(X_shifted, y_test)
print(f"Clean test accuracy  : {clean_acc:.4f}")
print(f"Shifted test accuracy: {shifted_acc:.4f}")
print(f"Drop                 : {clean_acc - shifted_acc:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
axes[0].imshow(X_test[0, 0], cmap="gray", vmin=0, vmax=1)
axes[0].set_title(f"clean, pred={model.predict(X_test[:1])[0]}")
axes[1].imshow(X_shifted[0, 0], cmap="gray", vmin=0, vmax=1)
axes[1].set_title(f"shifted, pred={model.predict(X_shifted[:1])[0]}")
axes[2].imshow(np.abs(X_shifted[0, 0] - X_test[0, 0]), cmap="magma")
axes[2].set_title("absolute difference")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Exercise: learning-rate sweep

The code below trains three short runs with different learning rates. The
values are chosen so the cell always finishes quickly.

**TODO / questions**

1. Which learning rate reaches the lowest loss in six epochs?
2. What happens with `lr=0.2`? Is the loss curve stable?
3. Is the best training loss always the best test accuracy?

In [ ]:
sweep = []
for lr in [0.005, 0.01, 0.02]:
    candidate = ShelfCNN(n_classes=4, image_size=16, random_state=RANDOM_STATE)
    start = time.perf_counter()
    losses = candidate.fit(
        X_train, y_train, epochs=6, batch_size=32, lr=lr, momentum=0.9
    )
    duration = time.perf_counter() - start
    sweep.append((lr, losses[-1], candidate.score(X_test, y_test), duration))

print(f"{'lr':>8} {'final loss':>12} {'test acc':>10} {'seconds':>9}")
for lr, loss, acc, duration in sweep:
    print(f"{lr:>8.3f} {loss:>12.4f} {acc:>10.4f} {duration:>9.2f}")

## 9. Exercise: add a third convolution block

`Sequential` lets you assemble a custom architecture. The example below adds a
third conv + pool block, which reduces the 16x16 image to 2x2 before the dense
classifier.

**TODO:** try `width=8`, `width=16` or `width=32`, or change `kernel=5`.
Does a deeper model help on only 400 training images?

In [ ]:
width = 24
deep_model = Sequential(
    [
        Conv2D(1, 12, kernel=3, stride=1, padding=1, random_state=0),
        ReLU(),
        MaxPool2D(2, 2),
        Conv2D(12, width, kernel=3, stride=1, padding=1, random_state=1),
        ReLU(),
        MaxPool2D(2, 2),
        Conv2D(width, 32, kernel=3, stride=1, padding=1, random_state=2),
        ReLU(),
        MaxPool2D(2, 2),
        Flatten(),
        Dense(32 * 2 * 2, 48, random_state=3),
        ReLU(),
        Dense(48, 4, random_state=4),
    ],
    random_state=0,
)

deep_history = deep_model.fit(
    X_train, y_train, epochs=8, batch_size=32, lr=0.02, momentum=0.9
)
print(f"Deep model loss: {deep_history[0]:.3f} -> {deep_history[-1]:.3f}")
print(f"Deep model test accuracy: {deep_model.score(X_test, y_test):.4f}")

## 10. From CNN to object detection: an IoU helper

A classifier says *what* is in the crop. A detector must also say *where* it is.
The standard output is a bounding box `(x1, y1, x2, y2)`.

**Intersection over Union (IoU)** measures how well two boxes overlap:

```
IoU = area(intersection) / area(union)
```

YOLO predicts boxes and class scores on a grid of cells, then removes duplicate
detections with non-maximum suppression (NMS). A future module will cover the
full detector; this exercise only builds the IoU primitive.

**TODO:** rewrite `iou` for boxes in `(x, y, width, height)` format and check
that it gives the same result.

In [ ]:
def iou(box_a, box_b):
    """IoU for two boxes in (x1, y1, x2, y2) format."""
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    intersection = inter_w * inter_h

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0


box_a = (2, 2, 10, 10)
box_b = (6, 6, 14, 14)
box_c = (20, 20, 28, 28)

print("IoU(a, a) =", iou(box_a, box_a))
print("IoU(a, b) =", iou(box_a, box_b))
print("IoU(a, c) =", iou(box_a, box_c))

## 11. Challenges

1. **Data.** Change `noise` to `0.15` and retrain. Which classes become hardest?
2. **Architecture.** Remove the second pooling layer. Does the model overfit?
3. **Regularisation.** Add a small weight-decay term to `SGD` (subtract
   `weight_decay * param` from each gradient) and compare.
4. **Detection.** Extend the generator to return the true bounding box of each
   shape, then train a tiny model to regress `(x1, y1, x2, y2)` from the image.
   This is the first step from classification to YOLO-style detection.
5. **Evaluation.** Replace the single train/test split with 5-fold
   cross-validation and report mean and standard deviation of accuracy.

Keep the notebook fast: prefer a few short runs over one very long run.